# 99 — Summary: later axes (QN, Accelerated, Stochastic, Tensor, Distributed)

Quick comparative runs for remaining research axes.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic
from cubic_reg.solvers import cr, quasi_newton, accelerated, tensor, distributed, stochastic

%matplotlib inline

In [ ]:
p = Quadratic(n=40, condition=100.0, seed=0)
x0 = np.ones(p.dim)
methods = {
    "CR": cr.minimize(p, x0=x0, M=1.0, eps=1e-8),
    "QN-CR": quasi_newton.minimize(p, x0=x0, M=1.0, eps=1e-6, memory=10),
    "Acc-CR": accelerated.minimize(p, x0=x0, M=1.0, eps=1e-6),
    "Tensor": tensor.minimize(p, x0=x0, L=1.0, eps=1e-5, max_iter=40),
    "Dist-ARC": distributed.minimize(p, x0=x0, n_workers=4, sketch_rank=10, eps=1e-5),
}
plt.figure(figsize=(8, 4))
for name, r in methods.items():
    gap = np.maximum(np.asarray(r.history_f) - p.f_star, 1e-16)
    plt.semilogy(gap, label=f"{name} ({r.time_sec:.3f}s, ||g||={r.grad_norm:.1e})")
plt.legend(); plt.xlabel("iteration"); plt.ylabel(r"$f-f^*$"); plt.grid(True, alpha=0.3)
plt.title("Later axes on quadratic"); plt.show()

In [ ]:
prob = stochastic.make_synthetic_logistic(n_samples=800, n_features=40, seed=0)
r_sc = stochastic.minimize(prob, max_iter=60, batch_grad=128, batch_hess=48, M=1.0, eps=5e-3)
r_sgd = stochastic.minimize_sgd(prob, max_iter=200, batch=64, lr=0.2, eps=5e-3)
r_adam = stochastic.minimize_adam(prob, max_iter=200, batch=64, lr=0.05, eps=5e-3)
plt.figure(figsize=(8, 4))
for name, r in [("Subsampled CR", r_sc), ("SGD", r_sgd), ("Adam", r_adam)]:
    plt.semilogy(r.history_grad_norm, label=f"{name} t={r.time_sec:.2f}s")
plt.legend(); plt.xlabel("iteration"); plt.ylabel(r"$\|\nabla f\|$"); plt.title("Stochastic baselines")
plt.grid(True, alpha=0.3); plt.show()

In [ ]:
p = Quadratic(n=50, condition=30.0, seed=0)
results = distributed.speedup_experiment(p, worker_counts=[1, 2, 4, 8], sketch_rank=8, eps=1e-4, max_iter=40, network_latency_ms=0.5)
ws = [1, 2, 4, 8]
t1 = results[0].time_sec
plt.figure(figsize=(6, 3.5))
plt.plot(ws, [t1 / r.time_sec for r in results], "o-")
plt.xlabel("workers"); plt.ylabel("speedup vs 1 worker"); plt.title("Simulated distributed speedup")
plt.grid(True, alpha=0.3); plt.show()
for w, r in zip(ws, results):
    print(w, r.time_sec, r.message)